RAG Pipelines- Data Ingestion to Vector DB Pipeline


In [39]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [40]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: system-design.pdf
  ✓ Loaded 7 pages

Total documents loaded: 7


In [41]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m155 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'system-design', 'source': '../data/pdf/system-design.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'system-design.pdf', 'file_type': 'pdf'}, page_content='System-design(page\n \n1-2)\n \nand\n \nDBMS(page\n \n3-4)\n                                         \npage\n \n:\n    \n1\n                    \n \n \nLLD\n,\n \nalso\n \ncalled\n \nObject-Oriented\n \nDesign\n \n(OOD)\n \nor\n \nSoftware\n \nArchitecture\n \nDesign,\n \nconverts\n \nhigh-level\n \nsystem\n \nrequirements\n \ninto\n \nactionable,\n \nclean,\n \nmaintainable,\n \nand\n \nmodular\n \ncode.\n \nWhile\n \nHigh-Level\n \nDesign\n \n(HLD)\n \ndecides\n \nwhich\n \nservices,\n \ndatabases,\n \nand\n \ncaches\n \nto\n \nuse\n,\n \nLLD\n \ndefines\n \nthe\n \ninternal\n \nclasses,\n \ninterfaces,\n \nrelationships,\n \ndesign\n \npatterns,\n \nand\n \ndatabase\n \nschemas\n \nof\n \na

In [42]:
### Text splitting get into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [43]:
chunks=split_documents(all_pdf_documents)
chunks

Split 7 documents into 21 chunks

Example chunk:
Content: System-design(page
 
1-2)
 
and
 
DBMS(page
 
3-4)
                                         
page
 
:
    
1
                    
 
 
LLD
,
 
also
 
called
 
Object-Oriented
 
Design
 
(OOD)
 
or
 
So...
Metadata: {'producer': 'Skia/PDF m155 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'system-design', 'source': '../data/pdf/system-design.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'system-design.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m155 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'system-design', 'source': '../data/pdf/system-design.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'system-design.pdf', 'file_type': 'pdf'}, page_content='System-design(page\n \n1-2)\n \nand\n \nDBMS(page\n \n3-4)\n                                         \npage\n \n:\n    \n1\n                    \n \n \nLLD\n,\n \nalso\n \ncalled\n \nObject-Oriented\n \nDesign\n \n(OOD)\n \nor\n \nSoftware\n \nArchitecture\n \nDesign,\n \nconverts\n \nhigh-level\n \nsystem\n \nrequirements\n \ninto\n \nactionable,\n \nclean,\n \nmaintainable,\n \nand\n \nmodular\n \ncode.\n \nWhile\n \nHigh-Level\n \nDesign\n \n(HLD)\n \ndecides\n \nwhich\n \nservices,\n \ndatabases,\n \nand\n \ncaches\n \nto\n \nuse\n,\n \nLLD\n \ndefines\n \nthe\n \ninternal\n \nclasses,\n \ninterfaces,\n \nrelationships,\n \ndesign\n \npatterns,\n \nand\n \ndatabase\n \nschemas\n \nof\n \na

embedding And vectorStoreDB

In [44]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [45]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7625.20it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/h1/840gqx7x7ldcq8b44nytvb5c0000gn/T/ipykernel_43613/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [46]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 171


In [47]:
chunks

[Document(metadata={'producer': 'Skia/PDF m155 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'system-design', 'source': '../data/pdf/system-design.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'system-design.pdf', 'file_type': 'pdf'}, page_content='System-design(page\n \n1-2)\n \nand\n \nDBMS(page\n \n3-4)\n                                         \npage\n \n:\n    \n1\n                    \n \n \nLLD\n,\n \nalso\n \ncalled\n \nObject-Oriented\n \nDesign\n \n(OOD)\n \nor\n \nSoftware\n \nArchitecture\n \nDesign,\n \nconverts\n \nhigh-level\n \nsystem\n \nrequirements\n \ninto\n \nactionable,\n \nclean,\n \nmaintainable,\n \nand\n \nmodular\n \ncode.\n \nWhile\n \nHigh-Level\n \nDesign\n \n(HLD)\n \ndecides\n \nwhich\n \nservices,\n \ndatabases,\n \nand\n \ncaches\n \nto\n \nuse\n,\n \nLLD\n \ndefines\n \nthe\n \ninternal\n \nclasses,\n \ninterfaces,\n \nrelationships,\n \ndesign\n \npatterns,\n \nand\n \ndatabase\n \nschemas\n \nof\n \na

In [48]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]
# texts
## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 21 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.07it/s]

Generated embeddings with shape: (21, 384)
Adding 21 documents to vector store...
Successfully added 21 documents to vector store
Total documents in collection: 192


Retriever Pipeline From VectorStore

In [49]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [50]:
rag_retriever

In [54]:
rag_retriever.retrieve("Design classic LLD problems: Parking Lot, Rate Limiter library, LRU Cache")

Retrieving documents for query: 'Design classic LLD problems: Parking Lot, Rate Limiter library, LRU Cache'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_a7eaa826_7',
  'content': 'System-design(page\n \n1-2)\n \nand\n \nDBMS(page\n \n3-4)\n                                         \npage\n \n:\n    \n3\n                    \n \n \n●\n \nParking\n \nLot\n \nSystem:\n \nManaging\n \nmultiple\n \nvehicle\n \nsizes,\n \nspots,\n \nticketing,\n \nand\n \ndynamic\n \npricing\n \nstrategies.\n \n●\n \nIn-Memory\n \nCache\n \n(LRU\n \nCache):\n \nUsing\n \na\n \ndoubly\n \nlinked\n \nlist\n \n+\n \nhash\n \nmap\n \nwith\n \neviction\n \nlogic\n \nand\n \nconcurrency\n \nlocks.\n \n●\n \nMovie\n \nTicket\n \n/\n \nFlight\n \nBooking\n \n(BookMyShow):\n \nSeat\n \nallocation,\n \nlocking\n \nmechanics,\n \nsearch,\n \nand\n \nconcurrent\n \nreservation\n \nhandling.\n \n●\n \nElevator\n \nSystem:\n \nHandling\n \nmultiple\n \nrequests,\n \ndirection\n \nscheduling\n \nalgorithms,\n \nand\n \ndispatch\n \noptimization.\n \n●\n \nRate\n \nLimiter\n \nLibrary:\n \nImplementing\n \nToken\n \nBucket\n \nor\n \nSliding\n \nWindow\n \nalgor